In [1]:
!pip install python-dotenv langchain_community langchain pypdf2 pypdf chromadb langchain_groq

Defaulting to user installation because normal site-packages is not writeable
  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)


In [2]:
import os
import dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [3]:
loader = PyPDFLoader('mergedd_bn_en.pdf')
doc = loader.load()
doc

[Document(metadata={'source': 'mergedd_bn_en.pdf', 'page': 0}, page_content='Question Answer T ags Context Response Type Language V ariations Follow-Up Suggestions Follow-Up \nQuestion/Clarification\nWhat is PMS (Premenstrual \nSyndrome)?PMS refers to physical and emotional \nsymptoms like mood swings, bloating, and \nfatigue before periods.Health, Basics Explaining pre-\nperiod symptomsStatement English What is premenstrual syndrome? What causes PMS? How can I manage PMS?\nWhat is the normal age to start \nperiods?Most girls start their periods between ages 9 \nand 16.Basics, Puberty Addressing age-\nrelated queriesStatement English What age do girls get their first period? When do periods \nstart?Is it okay to start periods early or late?\nCan I swim during my period? Y es, you can swim during your period using \ntampons or menstrual cups for protection.Activity, Hygiene Addressing activity \nconcernsSuggestion English Is it safe to swim while on my period? What precautions should I 

In [4]:
len(doc)

28

In [5]:
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
import chromadb

In [6]:
client = chromadb.PersistentClient(path='vectordb')
collection = client.get_or_create_collection(name='probahini')

In [7]:
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\shahabuddin akhon hr\AppData\Local\Temp\ipykernel_10756\2213097150.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
C:\Users\shahabuddin akhon hr\AppData\Roaming\Python\Python312\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


C:\Users\shahabuddin akhon hr\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
embeddings = embedding_model.embed_documents([text.page_content for text in doc])

In [9]:
for i, text in enumerate(doc):
    print(i)
    collection.add(
        ids=[str(i)],  # Unique ID for each document
        documents=[text.page_content],
        embeddings=[embeddings[i]],
        metadatas=[text.metadata]  # Optionally, include metadata
    )

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27


In [10]:
retriever = collection.query(
    query_texts=["What is period"],  # Your query
    n_results=2  # Number of results you want
)
print(retriever)

{'ids': [['14', '13']], 'distances': [[1.03743675824751, 1.1451406677987548]], 'metadatas': [[{'page': 14, 'source': 'mergedd_bn_en.pdf'}, {'page': 13, 'source': 'mergedd_bn_en.pdf'}]], 'embeddings': None, 'documents': [['Question Answer T ags Context Response Type Language V ariations Follow-Up Suggestions Follow-Up \nQuestion/Clarification\nCan periods stop suddenly for a \nmonth and restart later?Y es, hormonal imbalances or stress can \ncause missed periods, but it’s best to consult \na doctor if it happens often.Health, \nAwareness Irregular periods InformativeEnglish\nWhy did my periods skip a month?Should I worry if my period starts again \nunexpectedly?\nHow do I know when my period \nis about to start?Look for signs like mood changes, cramps, or \ntender breasts, which indicate menstruation \nis near .Health, \nAwarenessRecognizing \nmenstrual signs InformativeEnglish\nWhat are the signs my period is coming? Should I track my cycle for better preparation?\nIs it normal to have

In [11]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0,
    groq_api_key='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',
    model="llama-3.1-8b-instant",
)

In [12]:
from langchain_core.prompts import PromptTemplate

# Define the template that takes in the user question and the retrieved document
template = """
        You are an expert in answering menstrual period related topics. The user has a question, and you will answer it based on the relevant information provided from the menstrual related data.

        User question: {user_question}
        Relevant information: {answer}

        Provide a specific and accurate response based on the relevant information above.(NO PREAMBLE)
"""
# Create the PromptTemplate
prompt_template = PromptTemplate.from_template(
    template
)

In [13]:
query_text = 'Are there herbal remedies for period pain?'

retriever = collection.query(
    query_texts=query_text,  # Your query
    n_results=3  # Number of results you want
).get('documents')

In [14]:
chain = prompt_template | llm
res = chain.invoke(input={'user_question': query_text, 'answer': retriever})
print(res.content)

Yes, there are herbal remedies for period pain. Natural remedies include using a heating pad, staying hydrated, practicing yoga, or taking magnesium-rich foods to ease cramps.


In [ ]:
!pip freeze > requirements.txt